In [94]:
import pandas as pd                                                                 
import glob                                                                         
import re 

In [96]:
# Load and save as parquet                                                          
files = glob.glob('../data/raw/luka_*.csv')
                                            
dfs = []                                                                            
for f in files:                         
    year = re.search(r'(\d{4}_\d{2})', f).group(1)                                  
    df = pd.read_csv(f, skiprows=0)                                                 
                                    
    # BBRef CSVs have junk rows — drop rows where Rk is not numeric                 
    df = df[pd.to_numeric(df['Rk'], errors='coerce').notna()]      
    df['SEASON'] = year                                      
                                                                                    
    # Save parquet                                                                  
    out = f'../data/processed/luka_csv_{year}.parquet'                              
    df.to_parquet(out, index=False)                                                 
    print(f"Saved {out}")                   
    dfs.append(df)                                                                  
                
df_all = pd.concat(dfs, ignore_index=True)

Saved ../data/processed/luka_csv_2020_21.parquet
Saved ../data/processed/luka_csv_2018_19.parquet
Saved ../data/processed/luka_csv_2022_23.parquet
Saved ../data/processed/luka_csv_2024_25.parquet
Saved ../data/processed/luka_csv_2021_22.parquet
Saved ../data/processed/luka_csv_2023_24.parquet
Saved ../data/processed/luka_csv_2019_20.parquet


In [97]:
files = sorted(glob.glob('../data/raw/luka_*.csv'))                                 
                                                                                    
dfs = []                                                                            
for f in files:                                                                     
    year = re.search(r'(\d{4}_\d{2})', f).group(1)                                  
    df = pd.read_csv(f)                           
    df = df[pd.to_numeric(df['Rk'], errors='coerce').notna()].copy()                
    df.rename(columns={df.columns[5]: 'HOME_AWAY'}, inplace=True)   
    df['HOME'] = (df['HOME_AWAY'] != '@').astype(int)                               
    df['SEASON'] = year                              
    df['Date'] = pd.to_datetime(df['Date'])
    df['PTS'] = pd.to_numeric(df['PTS'], errors='coerce')                           
    df['TRB'] = pd.to_numeric(df['TRB'], errors='coerce')                           
    df['AST'] = pd.to_numeric(df['AST'], errors='coerce')                           
    df.to_parquet(f'../data/processed/luka_csv_{year}.parquet', index=False)        
    dfs.append(df)                                                                  
    print(f"{year}: {len(df)} games")   
                                                                                    
df_all = pd.concat(dfs,                                                             
ignore_index=True).sort_values('Date').reset_index(drop=True)                       
print(f"\nTotal: {len(df_all)} games")                                              
                                                                                    
# Basic analysis                            
print("\n--- Season Averages ---")                                                  
print(df_all.groupby('SEASON')[['PTS','TRB','AST']].mean().round(1))
                                                                                    
print("\n--- Home vs Away ---")         
print(df_all.groupby('HOME')[['PTS','TRB','AST']].mean().round(1))                  
                                                                                    
print("\n--- Overall Distribution ---")
print(df_all['PTS'].describe().round(1)) 

2018_19: 82 games
2019_20: 75 games
2020_21: 72 games
2021_22: 82 games
2022_23: 82 games
2023_24: 82 games
2024_25: 84 games

Total: 559 games

--- Season Averages ---
          PTS  TRB  AST
SEASON                 
2018_19  21.2  7.8  6.0
2019_20  28.8  9.4  8.8
2020_21  27.7  8.0  8.6
2021_22  28.4  9.1  8.7
2022_23  32.4  8.6  8.0
2023_24  33.9  9.2  9.8
2024_25  28.2  8.2  7.7

--- Home vs Away ---
       PTS  TRB  AST
HOME                
0     27.9  8.5  8.1
1     29.3  8.7  8.4

--- Overall Distribution ---
count    450.0
mean      28.6
std        9.0
min        0.0
25%       23.0
50%       28.5
75%       34.0
max       73.0
Name: PTS, dtype: float64


# Linear Model — Predicting Luka's Points per Game

**Goal:** predict `PTS` for an upcoming game using only information available *before* tip-off.

Features (all lagged/rolled to avoid leakage):
- `roll5_pts` — rolling 5-game avg of prior points  
- `roll5_fga` — rolling 5-game avg of prior FGA  
- `roll5_fg_pct` — rolling 5-game avg of prior FG%  
- `roll5_min` — rolling 5-game avg of prior minutes  
- `roll5_3pa` — rolling 5-game avg of prior 3-point attempts  
- `HOME` — 1 if home game, 0 if away  
- `days_rest` — days since previous game  
- `season_num` — integer year (2018 = 0, … 2024 = 6)

In [ ]:
import numpy as np

# ── 1. Parse minutes from "MM:SS" string to float ──────────────────────────
def parse_mp(mp_str):
    try:
        parts = str(mp_str).split(':')
        return int(parts[0]) + int(parts[1]) / 60
    except Exception:
        return np.nan

df_all['MIN_F'] = df_all['MP'].apply(parse_mp)

# ── 2. Ensure numeric columns ───────────────────────────────────────────────
for c in ['PTS', 'FGA', 'FG%', '3PA', 'MIN_F']:
    df_all[c] = pd.to_numeric(df_all[c], errors='coerce')

# ── 3. Sort chronologically ─────────────────────────────────────────────────
df_model = df_all.sort_values('Date').reset_index(drop=True).copy()

# ── 4. Rolling 5-game avg of PREVIOUS points only ──────────────────────────
# We only keep roll5_pts here. FGA, FG%, 3PA, and MIN all move together
# with points — including them causes multicollinearity (the model gets
# confused and assigns nonsense coefficients).
df_model['roll5_pts'] = (
    df_model['PTS'].shift(1).rolling(window=5, min_periods=3).mean()
)

# ── 5. Days rest ────────────────────────────────────────────────────────────
df_model['days_rest'] = df_model['Date'].diff().dt.days.clip(upper=14).fillna(2)

# ── 6. Lakers trade indicator ───────────────────────────────────────────────
# Luka was traded to the Lakers in Feb 2025. His role and scoring context
# changed significantly — we flag this so the model isn't blindly
# extrapolating his Dallas trend into a different team situation.
df_model['on_lakers'] = (
    (df_model['Date'] >= '2025-02-01') & (df_model['Team'] == 'LAL')
).astype(int)

FEATURES = ['roll5_pts', 'HOME', 'days_rest', 'on_lakers']

df_model.dropna(subset=FEATURES + ['PTS'], inplace=True)
df_model.reset_index(drop=True, inplace=True)

print(f"Model dataset: {len(df_model)} games")
print(f"Lakers games in dataset: {df_model['on_lakers'].sum()}")
df_model[FEATURES + ['PTS']].describe().round(2)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

X = df_model[FEATURES].values
y = df_model['PTS'].values

# Chronological 80/20 split — never shuffle time-series data,
# otherwise future games leak into training.
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {split_idx} games  →  up to {df_model['Date'].iloc[split_idx-1].date()}")
print(f"Test : {len(df_model)-split_idx} games  →  from {df_model['Date'].iloc[split_idx].date()}")

In [ ]:
# ── OLS (Ordinary Least Squares) via statsmodels ──────────────────────────
#
# OLS is the classic linear regression method. It finds the straight-line
# relationship between our features (X) and the target (PTS) by minimising
# the sum of squared errors — i.e. it picks coefficients that make the
# predictions as close to the actual values as possible.
#
# The model equation looks like:
#   PTS = b0 + b1*roll5_pts + b2*roll5_fga + ... + error
#
# We use statsmodels here (rather than just sklearn) because it gives us a
# full statistical summary: p-values, confidence intervals, and R² — not
# just predictions.

# add_constant() prepends a column of 1s to X_train so the model can fit
# an intercept (b0). Without it the regression line is forced through the
# origin, which is almost never what we want.
X_train_sm = sm.add_constant(X_train)

# Fit the OLS model on the training set.
# sm.OLS(y, X) sets up the model; .fit() actually runs the optimisation
# and returns a results object with all the statistics attached.
ols = sm.OLS(y_train, X_train_sm).fit()

# summary() prints the full regression table.
# Key things to read:
#   R-squared   — how much variance in PTS the model explains (0–1, higher = better)
#   coef        — each feature's coefficient: a +1 increase in that feature
#                 is associated with a +coef change in predicted PTS
#   P>|t|       — p-value; values < 0.05 mean the feature is statistically
#                 significant (unlikely to be noise)
#   [0.025/0.975] — 95% confidence interval for each coefficient
print(ols.summary(xname=['const'] + FEATURES))

In [ ]:
# ── sklearn model — evaluate on hold-out test set ─────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=== Test-set metrics ===")
print(f"  R²   : {r2:.3f}")
print(f"  RMSE : {rmse:.2f} pts")
print(f"  MAE  : {mae:.2f} pts")

# ── 5-fold time-series cross-validation ───────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5)
cv_rmse = []
for tr_idx, val_idx in tscv.split(X):
    m = LinearRegression().fit(X[tr_idx], y[tr_idx])
    preds = m.predict(X[val_idx])
    cv_rmse.append(np.sqrt(mean_squared_error(y[val_idx], preds)))

print(f"\n  CV RMSE (5-fold time-series): {np.mean(cv_rmse):.2f} ± {np.std(cv_rmse):.2f} pts")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Plot 1: Actual vs Predicted ────────────────────────────────────────────
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.5, edgecolors='none', color='steelblue', s=40)
lims = [min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2]
ax.plot(lims, lims, 'r--', lw=1.5, label='perfect')
ax.set_xlabel('Actual PTS')
ax.set_ylabel('Predicted PTS')
ax.set_title('Actual vs Predicted (test set)')
ax.legend()

# ── Plot 2: Residuals ──────────────────────────────────────────────────────
residuals = y_test - y_pred
ax = axes[1]
ax.scatter(y_pred, residuals, alpha=0.5, edgecolors='none', color='darkorange', s=40)
ax.axhline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Predicted PTS')
ax.set_ylabel('Residual (actual − pred)')
ax.set_title('Residuals vs Predicted')

# ── Plot 3: Coefficient bar chart ──────────────────────────────────────────
ax = axes[2]
coef_df = pd.Series(lr.coef_, index=FEATURES).sort_values()
colors = ['crimson' if c < 0 else 'steelblue' for c in coef_df]
coef_df.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Linear model coefficients')
ax.set_xlabel('Coefficient value')

plt.tight_layout()
plt.show()

# Improved Model — Adding Opponent Defensive Rating + Age

Two new features:
- **`opp_def_rating`** — the opponent team's defensive rating for that season, fetched from `nba_api`. Lower = better defence, so we expect a negative coefficient (harder defence → fewer points).
- **`age_at_game`** — Luka's exact age in decimal years on game date. A simple proxy for career stage — avoids needing multi-player data to fit a full trajectory curve.

In [ ]:
import time
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.static import teams as nba_teams

# Build two lookup dicts from the static teams table:
#   nba_abbr  -> team_id   (nba_api abbreviations, e.g. "PHX", "BKN")
#   team_id   -> nba_abbr
_static = nba_teams.get_teams()
NBA_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in _static}
ID_TO_NBA_ABBR = {t['id']: t['abbreviation'] for t in _static}

# BBRef uses slightly different abbreviations for some teams.
# Map BBRef abbr -> nba_api abbr so we can join on team_id.
BBREF_TO_NBA = {
    'CHO': 'CHA',   # Charlotte Hornets
    'PHO': 'PHX',   # Phoenix Suns
    'BRK': 'BKN',   # Brooklyn Nets
    'GSW': 'GSW',   # fine as-is but kept for clarity
    'NOP': 'NOP',
}

SEASONS = {
    '2018_19': '2018-19', '2019_20': '2019-20', '2020_21': '2020-21',
    '2021_22': '2021-22', '2022_23': '2022-23', '2023_24': '2023-24',
    '2024_25': '2024-25',
}

def fetch_def_ratings(season_str):
    """Return {team_id: def_rating} for one season."""
    df = leaguedashteamstats.LeagueDashTeamStats(
        season=season_str,
        measure_type_detailed_defense='Advanced',
    ).get_data_frames()[0]
    return dict(zip(df['TEAM_ID'], df['DEF_RATING']))

# Fetch once per season
def_rating_lookup = {}   # {season_key: {team_id: def_rating}}
for season_key, season_str in SEASONS.items():
    print(f"Fetching {season_str}...")
    def_rating_lookup[season_key] = fetch_def_ratings(season_str)
    time.sleep(0.6)

print("\nDone. 2023-24 sample (by team id):")
sample = sorted(def_rating_lookup['2023_24'].items(), key=lambda x: x[1])
print("Best defence:", [(ID_TO_NBA_ABBR.get(tid), r) for tid, r in sample[:3]])
print("Worst defence:", [(ID_TO_NBA_ABBR.get(tid), r) for tid, r in sample[-3:]])

In [ ]:
LUKA_DOB = pd.Timestamp('1999-02-28')

df_v2 = df_model.copy()

# ── Opponent defensive rating ───────────────────────────────────────────────
# Convert BBRef abbreviation -> nba_api abbreviation -> team_id -> def_rating
def get_def_rating(row):
    nba_abbr = BBREF_TO_NBA.get(row['Opp'], row['Opp'])
    team_id  = NBA_ABBR_TO_ID.get(nba_abbr)
    if team_id is None:
        return float('nan')
    return def_rating_lookup.get(row['SEASON'], {}).get(team_id, float('nan'))

df_v2['opp_def_rating'] = df_v2.apply(get_def_rating, axis=1)

# ── Age at game date (decimal years) ───────────────────────────────────────
df_v2['age_at_game'] = (df_v2['Date'] - LUKA_DOB).dt.days / 365.25

missing = df_v2['opp_def_rating'].isna().sum()
print(f"Missing opp_def_rating: {missing} games")
print(f"Age range: {df_v2['age_at_game'].min():.1f} – {df_v2['age_at_game'].max():.1f} years")
df_v2[['Date', 'Opp', 'opp_def_rating', 'age_at_game', 'PTS']].head(10)

In [ ]:
# on_lakers is dropped — all 25 Lakers games fall in the test set so the
# training data has zero Lakers observations. Including an all-zero column
# in training causes a near-singular matrix (condition number ~7e18) and
# produces numerically meaningless coefficients.
FEATURES_V2 = ['roll5_pts', 'HOME', 'days_rest', 'opp_def_rating', 'age_at_game']

df_v2.dropna(subset=FEATURES_V2 + ['PTS'], inplace=True)
df_v2.reset_index(drop=True, inplace=True)

X2 = df_v2[FEATURES_V2].values
y2 = df_v2['PTS'].values

split_idx2 = int(len(df_v2) * 0.8)
X2_train, X2_test = X2[:split_idx2], X2[split_idx2:]
y2_train, y2_test = y2[:split_idx2], y2[split_idx2:]

print(f"Dataset: {len(df_v2)} games")
print(f"Train: {split_idx2}  →  up to {df_v2['Date'].iloc[split_idx2-1].date()}")
print(f"Test : {len(df_v2)-split_idx2}  →  from {df_v2['Date'].iloc[split_idx2].date()}")

X2_train_sm = sm.add_constant(X2_train)
ols2 = sm.OLS(y2_train, X2_train_sm).fit()
print(ols2.summary(xname=['const'] + FEATURES_V2))

In [ ]:
lr2 = LinearRegression().fit(X2_train, y2_train)
y2_pred = lr2.predict(X2_test)

rmse2 = np.sqrt(mean_squared_error(y2_test, y2_pred))
mae2  = mean_absolute_error(y2_test, y2_pred)
r2_2  = r2_score(y2_test, y2_pred)

print("=== V2 Test-set metrics ===")
print(f"  R²   : {r2_2:.3f}")
print(f"  RMSE : {rmse2:.2f} pts")
print(f"  MAE  : {mae2:.2f} pts")

tscv2 = TimeSeriesSplit(n_splits=5)
cv2 = [np.sqrt(mean_squared_error(y2[v], LinearRegression().fit(X2[t], y2[t]).predict(X2[v])))
       for t, v in tscv2.split(X2)]
print(f"  CV RMSE: {np.mean(cv2):.2f} ± {np.std(cv2):.2f} pts")

# ── Plots ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.scatter(y2_test, y2_pred, alpha=0.5, edgecolors='none', color='steelblue', s=40)
lims = [min(y2_test.min(), y2_pred.min()) - 2, max(y2_test.max(), y2_pred.max()) + 2]
ax.plot(lims, lims, 'r--', lw=1.5, label='perfect')
ax.set_xlabel('Actual PTS'); ax.set_ylabel('Predicted PTS')
ax.set_title('Actual vs Predicted (V2)'); ax.legend()

ax = axes[1]
res2 = y2_test - y2_pred
ax.scatter(y2_pred, res2, alpha=0.5, edgecolors='none', color='darkorange', s=40)
ax.axhline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Predicted PTS'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Predicted (V2)')

ax = axes[2]
coef2 = pd.Series(lr2.coef_, index=FEATURES_V2).sort_values()
coef2.plot(kind='barh', ax=ax, color=['crimson' if c < 0 else 'steelblue' for c in coef2])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Coefficients (V2)'); ax.set_xlabel('Coefficient value')

plt.tight_layout()
plt.show()